In [61]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


# online retail

df = pd.read_excel("Raw_Online_Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [62]:
pd.set_option("display.max_columns", None)

In [63]:
df.dropna(inplace=True) #removing mssing values
df= df[df["Quantity"] > 0] #keeping only positive quantity
df= df[df["UnitPrice"] > 0] #keeping only positive quantity

df.to_csv("cleaned_online_retail.csv", index=False)

In [64]:
#Standardize Date Column
df['InvoiceDate'] = pd.to_datetime(df["InvoiceDate"])

## Feature Engineering

In [65]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Revenue"] = (df["Quantity"] * df["UnitPrice"]
)

### Extract date-time columns
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["MonthName"] = df["InvoiceDate"].dt.month_name()
df["DayName"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

#For monthly trend analysis:
df["YearMonth"] = (
    df["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)


df["CustomerID"] = df["CustomerID"].astype(str)

## Initial Data Exploration

In [66]:
# Number of rows
len(df)

# Number of transactions
df["InvoiceNo"].nunique()

# Number of products
df["StockCode"].nunique()

# Number of customers
df["CustomerID"].nunique()

# Number of countries
df["Country"].nunique()

# Total quantity sold
df["Quantity"].sum()

# Total revenue
df["Revenue"].sum()

np.float64(8911407.904)

### Create Core Business KPIs

In [67]:
total_revenue = df["Revenue"].sum()

total_orders = df["InvoiceNo"].nunique()

total_customers = df["CustomerID"].nunique()

total_products = df["StockCode"].nunique()

total_quantity = df["Quantity"].sum()

In [68]:
print("Revenue:", total_revenue)
print("Orders:", total_orders)
print("Customers:", total_customers)
print("Products:", total_products)
print("Units Sold:", total_quantity)

Revenue: 8911407.904
Orders: 18532
Customers: 4338
Products: 3665
Units Sold: 5167812


### Average Order Value

In [69]:
# One invoice represents an order.

# First calculate revenue per order.

order_revenue = (
    df.groupby("InvoiceNo")["Revenue"]
    .sum()
)

# Then:

average_order_value = order_revenue.mean()

average_order_value

# Or mathematically:
# average_order_value = total_revenue / total_orders

np.float64(480.8659563997409)

### Average items per order

In [70]:
average_items_per_order = (
    df.groupby("InvoiceNo")["Quantity"]
    .sum()
    .mean()
)

average_items_per_order

np.float64(278.8588387653788)

### Business Question 1: How Is Revenue Changing Over Time?

In [71]:
# Aggregate revenue by month.

monthly_revenue = (
    df.groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)
monthly_revenue

# Visualize:

fig = px.line(
    monthly_revenue,
    x="YearMonth",
    y="Revenue",
    markers=True,
    title="Monthly Revenue Trend"
)

fig.show()

### Business Question 2: Which Countries Generate the Most Revenue?

In [ ]:
country_sales = (
        df.groupby("Country")["Revenue"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

# Look at the top countries:

country_revenue.head(10)


,Country,Revenue
0,United Kingdom,7308391.554
1,Netherlands,285446.340
2,EIRE,265545.900
3,Germany,228867.140
4,France,209024.050
5,Australia,138521.310
6,Spain,61577.110
7,Switzerland,56443.950
8,Belgium,41196.340
9,Sweden,38378.330


In [73]:
# Visualization:

top_countries = country_revenue.head(10)

fig = px.bar(
    top_countries,
    x="Revenue",
    y="Country",
    orientation="h",
    title="Top 10 Countries by Revenue"
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})

fig.show()

In [74]:
#Important analytical improvement

#If the United Kingdom dominates the dataset, analyze countries excluding the UK too.

international = df[df["Country"] != "United Kingdom"]

#Then:

international_revenue = (
    international.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
international_revenue
#This often reveals patterns hidden by a dominant market.

,Country,Revenue
0,Netherlands,285446.34
1,EIRE,265545.90
2,Germany,228867.14
3,France,209024.05
4,Australia,138521.31
5,Spain,61577.11
6,Switzerland,56443.95
7,Belgium,41196.34
8,Sweden,38378.33
9,Japan,37416.37


In [75]:
fig = px.bar(
    international_revenue,
    x="Revenue",
    y="Country",
    orientation="h",
    title="Top 10 Countries by Revenue"
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})

fig.show()

### Business Question 3: Which Products Generate the Most Revenue?


In [76]:
product_revenue = (
    df.groupby("Description")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum")
    )
    .sort_values("Revenue", ascending=False)
    .reset_index()
)
product_revenue.head(10)

,Description,Revenue,Quantity
0,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
1,REGENCY CAKESTAND 3 TIER,142592.95,12402
2,WHITE HANGING HEART T-LIGHT HOLDER,100448.15,36725
3,JUMBO BAG RED RETROSPOT,85220.78,46181
4,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
5,POSTAGE,77803.96,3120
6,PARTY BUNTING,68844.33,15291
7,ASSORTED COLOUR BIRD ORNAMENT,56580.34,35362
8,Manual,53779.93,7173
9,RABBIT NIGHT LIGHT,51346.20,27202


In [77]:
top_products_revenue = product_revenue.head(10)

fig = px.bar(
    top_products_revenue,
    x="Revenue",
    y="Description",
    orientation="h",
    title="Top 10 Products by Revenue"
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})

fig.show()

### Business Question 4: Which Products Sell the Most Units?

In [78]:
# Revenue and quantity are not the same thing.
# A product can sell many units but have a low price.

product_quantity = (
    df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    product_quantity,
    x="Quantity",
    y="Description",
    orientation="h",
    title="Top 10 Products by Units Sold"
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})

fig.show()

### 14. Business Question 5: Which Customers Generate the Most Revenue?

In [ ]:
customer_revenue = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
customer_revenue.head(10)

# Chart:

top_customers = customer_revenue.head(10).copy()

top_customers["CustomerID"] = top_customers["CustomerID"].astype(str)

fig = px.bar(
    top_customers,
    x="CustomerID",
    y="Revenue",
    title="Top 10 Customers by Revenue"
)

fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

### Deep Question: How Dependent Is the Business on Its Top Customers?

In [80]:
# This is a much more interesting data science/business question.

# First calculate total customer revenue.
customer_revenue = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Calculate revenue contribution:
customer_share = customer_revenue / customer_revenue.sum() * 100

# Top 10 customer contribution:
customer_share.head(10).sum()

# You can also investigate the top 10%.
number_top_customers = int(len(customer_revenue) * 0.10)

top_10_percent_revenue = customer_revenue.head(
    number_top_customers
).sum()

# Percentage:
top_10_percent_share = (
    top_10_percent_revenue /
    customer_revenue.sum()
) * 100

top_10_percent_share

# Now students can answer:
# What percentage of company revenue comes from the top 10% of customers?
# That is a far stronger insight than just showing the top customers.

np.float64(61.334117110121674)

### Business Question 6: Which Days Generate the Most Revenue?

In [81]:

day_revenue = (
    df.groupby("DayName")["Revenue"]
    .sum()
    .reset_index()
)

# Days need proper ordering.

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_revenue["DayName"] = pd.Categorical(
    day_revenue["DayName"],
    categories=day_order,
    ordered=True
)

day_revenue = day_revenue.sort_values("DayName")

# Chart:

fig = px.bar(
    day_revenue,
    x="DayName",
    y="Revenue",
    title="Revenue by Day of the Week"
)

fig.show()

### 17. Business Question 7: What Time Do Customers Shop?

In [82]:
hourly_orders = (
    df.groupby("Hour")["InvoiceNo"]
    .nunique()
    .reset_index(name="Orders")
)

# Visualize:

fig = px.line(
    hourly_orders,
    x="Hour",
    y="Orders",
    markers=True,
    title="Orders by Hour of Day"
)

fig.show()

# This answers:

# What are the company's busiest shopping hours?

# This information could affect:

# Marketing campaign timing
# Customer support staffing
# Promotional notifications
# Website infrastructure planning

35. Dashboard Structure

Now turn all of this into one professional Streamlit dashboard.

I would structure it into these pages:

Overview
Sales Analysis
Product Analysis
Customer Analysis